In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train"
)

print(dataset[0])

/home/mohsinkhan/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using the latest cached version of the dataset since databricks/databricks-dolly-15k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/mohsinkhan/.cache/huggingface/datasets/databricks___databricks-dolly-15k/default/0.0.0/bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a (last modified on Mon Oct 19 21:10:32 2026).


{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [2]:
print(dataset)
print(dataset.column_names)
print(dataset[0])

Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})
['instruction', 'context', 'response', 'category']
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [3]:
from collections import Counter
print(Counter(dataset["category"]))

Counter({'open_qa': 3742, 'general_qa': 2191, 'classification': 2136, 'closed_qa': 1773, 'brainstorming': 1766, 'information_extraction': 1506, 'summarization': 1188, 'creative_writing': 709})


In [4]:
import pandas as pd

df = dataset.to_pandas()

print("Missing values:")
print(df.isnull().sum())

print("\nEmpty strings:")
print((df == "").sum())

Missing values:
instruction    0
context        0
response       0
category       0
dtype: int64

Empty strings:
instruction        0
context        10544
response           0
category           0
dtype: int64


In [5]:
df["instruction_len"] = df["instruction"].str.split().str.len()
df["response_len"] = df["response"].str.split().str.len()

print(df[["instruction_len", "response_len"]].describe())

       instruction_len  response_len
count     15011.000000  15011.000000
mean         12.378656     60.115515
std          17.048248     98.312539
min           1.000000      1.000000
25%           7.000000     13.000000
50%          10.000000     31.000000
75%          14.000000     73.000000
max         841.000000   4274.000000


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights: 100%|███████████████████████| 290/290 [00:00<00:00, 368.92it/s]


Model loaded successfully!


In [7]:
def generate_response(prompt):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

In [8]:
sample = dataset[0]

prompt = sample["instruction"]

if sample["context"].strip():
    prompt += "\nContext: " + sample["context"]

print("Instruction:", prompt)
print("Expected:", sample["response"])
print("Base model:", generate_response(prompt))

Instruction: When did Virgin Australia start operating?
Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.
Expected: Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.
Base model: According to the information provided, Virgin Australia started operating on 31 August 2000.


## Split the data

In [9]:
split_dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))

Training examples: 12008
Testing examples: 3003


In [10]:
import pandas as pd

eval_samples = test_dataset.select(range(50))

baseline_results = pd.DataFrame([
    {
        "instruction": ex["instruction"],
        "context": ex["context"],
        "expected": ex["response"],
    }
    for ex in eval_samples
])

print("Evaluation examples:", len(baseline_results))
print(baseline_results.head(3))

Evaluation examples: 50
                                         instruction context  \
0  Who were the children of the legendary Garth G...           
1  Give me a list of basic ingredients for baking...           
2      Write a funny and whimsical horoscope reading           

                                            expected  
0  Garth the Gardener, John the Oak, Gilbert of t...  
1  To bake cookies you will need flour, sugar, eg...  
2  The stars say you should be patient, and that ...  


## Test Baseline model wihtout finetuning

In [11]:
from tqdm.auto import tqdm

def build_prompt(example):
    prompt = example["instruction"]

    if example["context"].strip():
        prompt += "\n\nContext:\n" + example["context"]

    return prompt

baseline_results["prompt"] = baseline_results.apply(
    lambda row: build_prompt(row),
    axis=1
)

baseline_results["base_response"] = [
    generate_response(prompt)
    for prompt in tqdm(baseline_results["prompt"])
]

print(
    baseline_results[
        ["instruction", "expected", "base_response"]
    ].head(3)
)

100%|███████████████████████████████████████████| 50/50 [01:01<00:00,  1.22s/it]

                                         instruction  \
0  Who were the children of the legendary Garth G...   
1  Give me a list of basic ingredients for baking...   
2      Write a funny and whimsical horoscope reading   

                                            expected  \
0  Garth the Gardener, John the Oak, Gilbert of t...   
1  To bake cookies you will need flour, sugar, eg...   
2  The stars say you should be patient, and that ...   

                                       base_response  
0  In the fantasy series "A Song of Ice and Fire"...  
1  Sure! Here's a simple recipe for you to start ...  
2  It's hard to say what exactly you want the hor...  


In [12]:
baseline_results.to_csv("baseline_results.csv", index=False)

print("Baseline saved successfully!")

Baseline saved successfully!


## Prepare the training data

In [13]:
def format_example(example):
    prompt = example["instruction"]

    if example["context"].strip():
        prompt += "\n\nContext:\n" + example["context"]

    messages = [
        {
            "role": "user",
            "content": prompt
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

print(format_example(train_dataset[0]))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a digital camera?<|im_end|>
<|im_start|>assistant
A digital camera is a hand held device that captures images with a push of a button.  These images are usually stored on a memory disk or card which can be downloaded to a computer at a later time.  Digital cameras were also part of a revolution that replaced photographic film.  Digital cameras use flashes when no natural light is available.  They produce better picture quality than traditional photographic film cameras and there is no cost to develop pictures and print them in order to enjoy them.<|im_end|>



In [14]:
text = format_example(train_dataset[0])

tokens = tokenizer(
    text,
    truncation=True,
    max_length=512
)

print("Token count:", len(tokens["input_ids"]))
print("First 20 token IDs:", tokens["input_ids"][:20])

Token count: 133
First 20 token IDs: [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645]


In [15]:
def tokenize_example(example):
    text = format_example(example)

    return tokenizer(
        text,
        truncation=True,
        max_length=512
    )

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_example,
    remove_columns=test_dataset.column_names
)

print(tokenized_train)
print(tokenized_test)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 12008
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 3003
})


## Configure LoRA for our Qwen model

In [16]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [17]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator ready!")

Data collator ready!


In [18]:
import trl
print(trl.__version__)

1.16.0


In [19]:
import inspect
from trl import SFTTrainer

print(inspect.signature(SFTTrainer))

(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[Any]], dict[str, Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None = None, optimiz

In [20]:
from trl import SFTConfig
import inspect

print(inspect.signature(SFTConfig))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, Any] | str | None = None, torch_compile: bool = False, torch_compile_ba

In [21]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [22]:
print(tokenizer.apply_chat_template(
    [
        {"role": "user", "content": "What is AI?"},
        {"role": "assistant", "content": "AI is artificial intelligence."}
    ],
    tokenize=False
))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is AI?<|im_end|>
<|im_start|>assistant
AI is artificial intelligence.<|im_end|>



In [23]:
import inspect
from trl import SFTConfig

print("completion_only_loss supported:",
      "completion_only_loss" in inspect.signature(SFTConfig).parameters)

completion_only_loss supported: True


In [24]:
def format_example(example):
    prompt = example["instruction"]

    if example["context"].strip():
        prompt += "\n\nContext:\n" + example["context"]

    return {
        "prompt": tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True
        ),
        "completion": example["response"] + tokenizer.eos_token
    }

print(format_example(train_dataset[0]))

{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is a digital camera?<|im_end|>\n<|im_start|>assistant\n', 'completion': 'A digital camera is a hand held device that captures images with a push of a button.  These images are usually stored on a memory disk or card which can be downloaded to a computer at a later time.  Digital cameras were also part of a revolution that replaced photographic film.  Digital cameras use flashes when no natural light is available.  They produce better picture quality than traditional photographic film cameras and there is no cost to develop pictures and print them in order to enjoy them.<|im_end|>'}


In [25]:
formatted_train = train_dataset.map(
    format_example,
    remove_columns=train_dataset.column_names
)

print(formatted_train[0].keys())
print(formatted_train[0]["prompt"][:200])
print(formatted_train[0]["completion"][:200])

dict_keys(['prompt', 'completion'])
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a digital camera?<|im_end|>
<|im_start|>assistant

A digital camera is a hand held device that captures images with a push of a button.  These images are usually stored on a memory disk or card which can be downloaded to a computer at a later time.  D


In [26]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./dolly-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    completion_only_loss=True,
    assistant_only_loss=False,
    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [27]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    processing_class=tokenizer
)

In [28]:
train_result = trainer.train()

Step,Training Loss
10,2.589911
20,2.220674
30,2.186472
40,2.293574
50,2.376878
60,2.278240
70,2.166441
80,2.289018
90,2.258177
100,2.185444


In [29]:
model.save_pretrained("./dolly-lora/final_adapter")
tokenizer.save_pretrained("./dolly-lora/final_adapter")

print("LoRA adapter saved successfully!")

LoRA adapter saved successfully!


In [30]:
from tqdm.auto import tqdm

finetuned_results = baseline_results.copy()

finetuned_results["finetuned_response"] = [
    generate_response(prompt)
    for prompt in tqdm(finetuned_results["prompt"])
]

finetuned_results.to_csv("finetuned_results.csv", index=False)

print("Fine-tuned evaluation completed!")

  0%|                                                    | 0/50 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.
100%|███████████████████████████████████████████| 50/50 [02:09<00:00,  2.59s/it]

Fine-tuned evaluation completed!


In [31]:
import pandas as pd

comparison = pd.read_csv("finetuned_results.csv")

comparison[
    ["instruction", "expected", "base_response", "finetuned_response"]
].head(10)

,instruction,expected,base_response,finetuned_response
0,Who were the children of the legendary Garth G...,"Garth the Gardener, John the Oak, Gilbert of t...","In the fantasy series ""A Song of Ice and Fire""...",The shmediz- ]主营淡化/deading/deicing引力density圭ận...
1,Give me a list of basic ingredients for baking...,"To bake cookies you will need flour, sugar, eg...",Sure! Here's a simple recipe for you to start ...,1淡化/deicingiz- ]主营- ]ading引力density圭ận应对ajdev...
2,Write a funny and whimsical horoscope reading,"The stars say you should be patient, and that ...",It's hard to say what exactly you want the hor...,I-淡化/deicing引力density主营- ]ading穆圭ận调剂izmediziz...
3,Who was the first person to do spacewalk?,"Alexei Leonov, a soviet cosmonaut was the firs...",The concept of spacewalks is not new and has b...,Neiliz-淡化/deicing引力density主营- ] shve-liteading...
4,What causes an airplane wing to fly?,"As Bernoulli's principal explains, because of ...",An airplane wing is designed and optimized for...,Aniz-淡化/deicing引力density主营- ] shve-liteading/d...
5,Could a nuclear explosion change the Earth's r...,Nuclear explosions release more energy than an...,A nuclear explosion would not have any measura...,A淡化/deicing引力- ]主营izve-lite shmedizizizmapadin...
6,"In music theory, what is the circle of fifths?",The circle of fifths is a way of organizing th...,"The ""circle of fifths"" in music refers to the ...",Theiz-淡化/deading主营- ] shve-liteajde淡化圭ậnicing引...
7,What should I do on a free afternoon in San Fr...,You can visit and take photos at San Francisco...,"As an AI language model, I don't have personal...",Ifiz-淡化/deicing引力density圭ading穆ajde淡化主营izon-li...
8,Extract the dates from the following.,"February 16, 2023",The extract contains no dates. It appears to b...,Theiz-淡化主营izon/deicing引力density圭ận应对ajdeve-lit...
9,As per the passage whose works did Narendranat...,"David Hume, Immanuel Kant, Johann Gottlieb Fic...",Narendra Nandini studied the works of David Hu...,N淡化主营- ]ading/deicing引力density圭ận调剂穆ajizmedizv...


In [32]:
trainer.state.log_history[-10:]

[{'loss': 2.239423942565918,
  'grad_norm': 1.46875,
  'learning_rate': 1.1772853185595568e-05,
  'entropy': 1.9183404121547938,
  'mean_token_accuracy': 0.5766106724739075,
  'num_tokens': 1844325.0,
  'epoch': 0.941991341991342,
  'step': 1360},
 {'loss': 2.3867586135864256,
  'grad_norm': 1.71875,
  'learning_rate': 1.038781163434903e-05,
  'entropy': 1.9651995178312063,
  'mean_token_accuracy': 0.5831316456198692,
  'num_tokens': 1857743.0,
  'epoch': 0.9489177489177489,
  'step': 1370},
 {'loss': 2.160616874694824,
  'grad_norm': 1.3125,
  'learning_rate': 9.002770083102493e-06,
  'entropy': 1.948889967612922,
  'mean_token_accuracy': 0.5847305422648787,
  'num_tokens': 1872684.0,
  'epoch': 0.9558441558441558,
  'step': 1380},
 {'loss': 2.262212944030762,
  'grad_norm': 1.625,
  'learning_rate': 7.617728531855956e-06,
  'entropy': 2.039408157765865,
  'mean_token_accuracy': 0.5704389184713363,
  'num_tokens': 1884425.0,
  'epoch': 0.9627705627705627,
  'step': 1390},
 {'loss': 2.

In [33]:
print("Model type:", type(model))
print("Adapter enabled:", model.active_adapters)

prompt = "What is machine learning?"
print("\nResponse:")
print(generate_response(prompt))

Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Adapter enabled: ['default']

Response:
Machine-淡化/deicingizveading/de主营- ] shmediziziziz- ],引力density圭ận应对ajdeizmapiziziz-淡化穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆调剂izon-liteduceallisar淡化主营- where MAPabsve淡化/deicing/deicing/deicing/deading/deicing/deading/deicing/deicing/deicing/deicing/deicing/deading/deicing/deicing/de


In [34]:
model.eval()

with model.disable_adapter():
    print(generate_response("What is machine learning?"))

Machine learning is a type of artificial intelligence that enables computers to learn from data and improve their performance on specific tasks or problems. It involves the use of algorithms and statistical models to enable machines to make decisions and take actions based on patterns in data.
In simple terms, it's like teaching a computer how to recognize patterns in data and then using those patterns to make predictions or take decisions. Machine learning can be used for a wide range of applications, including image recognition, natural language processing, autonomous vehicles,


In [35]:
import torch

for name, param in model.named_parameters():
    if "lora_" in name:
        print(
            name,
            "| mean:", param.data.float().mean().item(),
            "| std:", param.data.float().std().item(),
            "| finite:", torch.isfinite(param.data).all().item()
        )

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight | mean: 0.00022194131452124566 | std: 0.019883982837200165 | finite: True
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight | mean: -8.006860298337415e-06 | std: 0.005426635965704918 | finite: True
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight | mean: -0.00018094493134412915 | std: 0.019593728706240654 | finite: True
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight | mean: -0.0001853498979471624 | std: 0.004505752585828304 | finite: True
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight | mean: 0.0004014958976767957 | std: 0.020100178197026253 | finite: True
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight | mean: -5.2306513680377975e-05 | std: 0.006034380756318569 | finite: True
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight | mean: 0.00017046774155460298 | std: 0.020006680861115456

In [36]:
print("PROMPT:")
print(formatted_train[0]["prompt"])

print("\nCOMPLETION:")
print(formatted_train[0]["completion"])

PROMPT:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a digital camera?<|im_end|>
<|im_start|>assistant


COMPLETION:
A digital camera is a hand held device that captures images with a push of a button.  These images are usually stored on a memory disk or card which can be downloaded to a computer at a later time.  Digital cameras were also part of a revolution that replaced photographic film.  Digital cameras use flashes when no natural light is available.  They produce better picture quality than traditional photographic film cameras and there is no cost to develop pictures and print them in order to enjoy them.<|im_end|>


In [37]:
print("Model dtype:", model.dtype)
print("Compute dtype:", model.config.torch_dtype)
print("Training FP16:", trainer.args.fp16)
print("Training BF16:", trainer.args.bf16)
print("4-bit quantization:", model.config.quantization_config)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Model dtype: torch.bfloat16
Compute dtype: torch.bfloat16
Training FP16: False
Training BF16: False
4-bit quantization: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [38]:
batch = trainer.data_collator([trainer.train_dataset[0]])

print("Batch keys:", batch.keys())

labels = batch["labels"][0]
valid_labels = labels[labels != -100]

print("Supervised tokens:", valid_labels.numel())
print("\nText used for loss:")
print(tokenizer.decode(valid_labels, skip_special_tokens=False)[:500])

Batch keys: dict_keys(['input_ids', 'labels', 'attention_mask'])
Supervised tokens: 97

Text used for loss:
A digital camera is a hand held device that captures images with a push of a button.  These images are usually stored on a memory disk or card which can be downloaded to a computer at a later time.  Digital cameras were also part of a revolution that replaced photographic film.  Digital cameras use flashes when no natural light is available.  They produce better picture quality than traditional photographic film cameras and there is no cost to develop pictures and print them in order to enjoy th


In [39]:
for log in trainer.state.log_history:
    if "loss" in log:
        print(
            f"Step: {log.get('step')}, "
            f"Loss: {log['loss']}"
        )

Step: 10, Loss: 2.5899112701416014
Step: 20, Loss: 2.2206741333007813
Step: 30, Loss: 2.186471939086914
Step: 40, Loss: 2.293574333190918
Step: 50, Loss: 2.376878356933594
Step: 60, Loss: 2.2782403945922853
Step: 70, Loss: 2.1664405822753907
Step: 80, Loss: 2.2890180587768554
Step: 90, Loss: 2.258176803588867
Step: 100, Loss: 2.185444450378418
Step: 110, Loss: 2.1899614334106445
Step: 120, Loss: 2.1692167282104493
Step: 130, Loss: 2.016992378234863
Step: 140, Loss: 2.307510757446289
Step: 150, Loss: 1.93599853515625
Step: 160, Loss: 2.1237478256225586
Step: 170, Loss: 2.3741621017456054
Step: 180, Loss: 2.0495075225830077
Step: 190, Loss: 2.2386011123657226
Step: 200, Loss: 2.234309959411621
Step: 210, Loss: 1.6957630157470702
Step: 220, Loss: 2.2017915725708006
Step: 230, Loss: 2.1265939712524413
Step: 240, Loss: 2.1681192398071287
Step: 250, Loss: 2.226205062866211
Step: 260, Loss: 2.1247268676757813
Step: 270, Loss: 1.924418830871582
Step: 280, Loss: 2.0510982513427733
Step: 290, Lo

In [40]:
for name, param in model.named_parameters():
    if "lora_" in name and "lora_B" in name:
        print(
            name,
            "mean:", param.data.float().mean().item(),
            "std:", param.data.float().std().item(),
            "max abs:", param.data.float().abs().max().item()
        )
        break

base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight mean: -8.006860298337415e-06 std: 0.005426635965704918 max abs: 0.019287109375


In [41]:
for name, param in model.named_parameters():
    if "lora_" not in name:
        print(name, "requires_grad:", param.requires_grad)
        break

base_model.model.model.embed_tokens.weight requires_grad: False


In [42]:
prompt = "What is machine learning?"

model.eval()

# Test base model
with model.disable_adapter():
    base_output = generate_response(prompt)

# Adapter is automatically restored here
tuned_output = generate_response(prompt)

print("BASE MODEL:\n", base_output)
print("\nFINE-TUNED MODEL:\n", tuned_output)

BASE MODEL:
 Machine learning is a type of artificial intelligence that enables computers to learn from data and improve their performance on specific tasks or problems. It involves the use of algorithms and statistical models to enable machines to make decisions and take actions based on patterns in data.
In simple terms, it's like teaching a computer how to recognize patterns in data and then using those patterns to make predictions or take decisions. Machine learning can be used for a wide range of applications, including image recognition, natural language processing, autonomous vehicles,

FINE-TUNED MODEL:
 Machine learning is the study of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. It involves training a computer program on a large dataset to identify patterns or make predictions based on those patterns. Machine learning can be used for a variety of purposes such as image recognition, natural language processing,

In [43]:
prompts = [
    "What is machine learning?",
    "Explain overfitting in simple terms.",
    "What is the difference between AI and ML?",
    "Write a Python function to reverse a string.",
    "Why is data preprocessing important?"
]

model.eval()

for prompt in prompts:
    with model.disable_adapter():
        base = generate_response(prompt)

    tuned = generate_response(prompt)

    print(f"\nPROMPT: {prompt}")
    print(f"BASE: {base[:250]}")
    print(f"TUNED: {tuned[:250]}")
    print("-" * 60)


PROMPT: What is machine learning?
BASE: Machine learning is a type of artificial intelligence that enables computers to learn from data and improve their performance on specific tasks or problems. It involves the use of algorithms and statistical models to enable machines to make decisions
TUNED: Machine learning is the study of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. It involves training a computer program on a large dataset to identify patterns or make predictions based
------------------------------------------------------------

PROMPT: Explain overfitting in simple terms.
BASE: Overfitting is when a model performs well on the training data but poorly on unseen test data. It's like giving your computer a really good math problem and then telling it to solve it again and again. But instead of solving the same problem every ti
TUNED: Overfitting is when a model performs well on the training data but poorly on un

In [44]:
import math

for p in [0.1, 0.5, 0.9]:
    loss = -math.log(p)
    print(f"Probability: {p}, Loss: {loss:.4f}")

Probability: 0.1, Loss: 2.3026
Probability: 0.5, Loss: 0.6931
Probability: 0.9, Loss: 0.1054


In [45]:
import pandas as pd

base = pd.read_csv("baseline_results.csv")
tuned = pd.read_csv("finetuned_results.csv")

print("Base results:", base.shape)
print("Fine-tuned results:", tuned.shape)

print("\nBase columns:", base.columns.tolist())
print("Fine-tuned columns:", tuned.columns.tolist())

Base results: (50, 5)
Fine-tuned results: (50, 6)

Base columns: ['instruction', 'context', 'expected', 'prompt', 'base_response']
Fine-tuned columns: ['instruction', 'context', 'expected', 'prompt', 'base_response', 'finetuned_response']


In [46]:
import pandas as pd

df = pd.read_csv("finetuned_results.csv")

for i in range(5):
    row = df.iloc[i]

    print(f"\n{'=' * 60}")
    print("INSTRUCTION:", row["instruction"])
    print("\nEXPECTED:", str(row["expected"])[:300])
    print("\nBASE:", str(row["base_response"])[:300])
    print("\nFINE-TUNED:", str(row["finetuned_response"])[:300])


INSTRUCTION: Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?

EXPECTED: Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree

BASE: In the fantasy series "A Song of Ice and Fire" by George R.R. Martin, there is no character named "Garth Greenhand" or "The High King of the First Men." The characters you might be thinking of could include:
- Aria (a young girl)
- Joffrey Baratheon
- Sansa Linenweaver
- Arya Stark
- Sansa's mother,

FINE-TUNED: The shmediz- ]主营淡化/deading/deicing引力density圭ận应对ajde淡化穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆调剂穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆穆

INSTRUCTION: Give me a list of basic ingredients for baking cookies

EXPECTED: To bake cookies you will need flour, sugar, egg

In [47]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_name = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./dolly-lora/final_adapter"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

fresh_base = AutoModelForCausalLM.from_pretrained(
    base_name,
    quantization_config=quant_config,
    device_map="auto"
)

fresh_model = PeftModel.from_pretrained(
    fresh_base,
    adapter_path
)

fresh_model.eval()

Loading weights: 100%|███████████████████████| 290/290 [00:00<00:00, 377.54it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear4bit(

In [49]:
def generate_with_fresh_model(prompt):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(fresh_model.device)

    with torch.no_grad():
        outputs = fresh_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

print(generate_with_fresh_model("What is machine learning?"))

Machine learning is the study of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. It involves training a computer program on a large dataset to identify patterns or make predictions based on those patterns. The goal is to create systems that can automatically improve their performance as more data is fed into them.


In [50]:
import pandas as pd

results = []

fresh_model.eval()

for example in eval_samples:
    prompt = example["instruction"]

    if example["context"].strip():
        prompt += "\n\nContext:\n" + example["context"]

    with fresh_model.disable_adapter():
        base_response = generate_with_fresh_model(prompt)

    tuned_response = generate_with_fresh_model(prompt)

    results.append({
        "instruction": example["instruction"],
        "expected": example["response"],
        "base_response": base_response,
        "finetuned_response": tuned_response
    })

evaluation_df = pd.DataFrame(results)
evaluation_df.to_csv("test_evaluation.csv", index=False)

print("Evaluated examples:", len(evaluation_df))
print(evaluation_df.head(2).to_string(index=False))

Evaluated examples: 50
                                                                                                                 instruction                                                                                                                                                                                                                                                                                                                 expected                                                                                                                                                                                                                                                                                                                                                                 base_response                                                                                                                                                                              

In [51]:
import pandas as pd

df = pd.read_csv("test_evaluation.csv")

for col in ["base_response", "finetuned_response"]:
    df[col] = df[col].fillna("")

print("Total test examples:", len(df))

print("\nEmpty responses:")
print("Base:", (df["base_response"].str.strip() == "").sum())
print("Fine-tuned:", (df["finetuned_response"].str.strip() == "").sum())

print("\nAverage response length (characters):")
print("Base:", df["base_response"].str.len().mean())
print("Fine-tuned:", df["finetuned_response"].str.len().mean())

print("\nFirst 10 instructions:")
for i, row in df.head(10).iterrows():
    print(f"\n{i+1}. {row['instruction']}")
    print("BASE:", row["base_response"][:150])
    print("TUNED:", row["finetuned_response"][:150])

Total test examples: 50

Empty responses:
Base: 0
Fine-tuned: 0

Average response length (characters):
Base: 374.42
Fine-tuned: 218.6

First 10 instructions:

1. Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?
BASE: In the fantasy series "A Song of Ice and Fire" by George R.R. Martin, there is no character named "Garth Greenhand" or "The High King of the First Men
TUNED: The children of the legendary Garth Greenhand, the High King of the First Men, were named Gwendolyn, Gwendolyn's daughter, and Gwendolyn's daughter's 

2. Give me a list of basic ingredients for baking cookies
BASE: Sure! Here's a simple recipe for you to start with:

**Cookies Recipe**

### Ingredients:
- 2 cups of all-purpose flour (or any type of flour)
- 1/4 c
TUNED: 1. Flour
2. Sugar
3. Butter
4. Salt
5. Baking soda
6. Yeast
7. Chocolate chips
8. Cocoa powder
9. Peanut butter
10. Chia seeds
11. Almond
12. Cinnamon

3. Write a funny and wh

In [52]:
for i, row in evaluation_df.iterrows():
    print(f"\n{'='*70}")
    print(f"Example {i+1}: {row['instruction']}")
    print(f"\nEXPECTED:\n{row['expected'][:350]}")
    print(f"\nBASE:\n{row['base_response'][:350]}")
    print(f"\nFINE-TUNED:\n{row['finetuned_response'][:350]}")


Example 1: Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?

EXPECTED:
Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree

BASE:
In the fantasy series "A Song of Ice and Fire" by George R.R. Martin, there is no character named "Garth Greenhand" or "The High King of the First Men." The characters you might be thinking of could include:
- Aria (a young girl)
- Joffrey Baratheon
- Sansa Linenweaver
- Arya Stark
- Sansa's mother, who was probably also named Sansa
- Sansa's fathe

FINE-TUNED:
The children of the legendary Garth Greenhand, the High King of the First Men, were named Gwendolyn, Gwendolyn's daughter, and Gwendolyn's daughter's daughter.

Example 2: Give me a list of basic ingredients for bak

In [53]:
print("Current epoch:", trainer.state.epoch)
print("Current steps:", trainer.state.global_step)
print("Configured epochs:", trainer.args.num_train_epochs)

Current epoch: 1.0
Current steps: 1444
Configured epochs: 1


In [54]:
trainer.model.save_pretrained("./dolly-lora/epoch1_backup")
tokenizer.save_pretrained("./dolly-lora/epoch1_backup")

('./dolly-lora/epoch1_backup/tokenizer_config.json',
 './dolly-lora/epoch1_backup/chat_template.jinja',
 './dolly-lora/epoch1_backup/tokenizer.json')

In [55]:
trainer.args.num_train_epochs = 2
trainer.train()

Step,Training Loss
10,2.361398
20,2.074718
30,2.086075
40,2.199598
50,2.281003
60,2.193574
70,2.101983
80,2.212366
90,2.195971
100,2.126842


TrainOutput(global_step=2888, training_loss=2.1016977381508104, metrics={'train_runtime': 3601.2788, 'train_samples_per_second': 6.414, 'train_steps_per_second': 0.802, 'total_flos': 8421688413186048.0, 'train_loss': 2.1016977381508104, 'entropy': 1.9979452010123961, 'mean_token_accuracy': 0.5616156824173466, 'num_tokens': 5873868.0, 'epoch': 2.0})

In [56]:
trainer.model.save_pretrained("./dolly-lora/epoch2")
tokenizer.save_pretrained("./dolly-lora/epoch2")

('./dolly-lora/epoch2/tokenizer_config.json',
 './dolly-lora/epoch2/chat_template.jinja',
 './dolly-lora/epoch2/tokenizer.json')

In [57]:
import os

adapter_path = "./dolly-lora/epoch2"

print(os.listdir(adapter_path))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [58]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    quantization_config=quant_config,
    device_map="auto"
)

model_epoch2 = PeftModel.from_pretrained(
    base_model,
    "./dolly-lora/epoch2"
)

model_epoch2.eval()
print("Adapter loaded successfully!")

Loading weights: 100%|███████████████████████| 290/290 [00:00<00:00, 443.14it/s]


Adapter loaded successfully!


In [59]:
prompt = "Explain machine learning in simple terms."

messages = [{"role": "user", "content": prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model_epoch2.device)

with torch.no_grad():
    # Fine-tuned model
    output = model_epoch2.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("Epoch-2 model:", response)

Epoch-2 model: Machine learning is the process of training a computer to make decisions based on data that it has been trained with. It can be used for many different purposes such as image recognition, natural language processing, and speech recognition. Machine learning algorithms use statistical models to identify patterns in data and then predict what will happen next.


In [60]:
with model_epoch2.disable_adapter():
    with torch.no_grad():
        base_output = model_epoch2.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

base_response = tokenizer.decode(
    base_output[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("BASE MODEL:\n", base_response)
print("\nEPOCH-2 MODEL:\n", response)

BASE MODEL:
 Machine learning is like teaching your computer to learn! It's like having a smart cookie that learns and improves on its own without being told what to do. So, instead of telling it what to do, the cookie just keeps trying different things until it finds one that works best for you. This way, it can get better at doing whatever task we want it to do! And because it's self-learning, it doesn't need to be told exactly how to do something - it just knows when it

EPOCH-2 MODEL:
 Machine learning is the process of training a computer to make decisions based on data that it has been trained with. It can be used for many different purposes such as image recognition, natural language processing, and speech recognition. Machine learning algorithms use statistical models to identify patterns in data and then predict what will happen next.


In [61]:
results_epoch2 = []

model_epoch2.eval()

for example in eval_samples:
    prompt = example["instruction"]

    if example["context"].strip():
        prompt += "\n\nContext:\n" + example["context"]

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model_epoch2.device)

    with model_epoch2.disable_adapter():
        with torch.no_grad():
            base_output = model_epoch2.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False
            )

    with torch.no_grad():
        tuned_output = model_epoch2.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    results_epoch2.append({
        "instruction": example["instruction"],
        "expected": example["response"],
        "base_response": tokenizer.decode(
            base_output[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ),
        "epoch2_response": tokenizer.decode(
            tuned_output[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )
    })

evaluation_epoch2 = pd.DataFrame(results_epoch2)
evaluation_epoch2.to_csv("epoch2_evaluation.csv", index=False)

print("Evaluated examples:", len(evaluation_epoch2))
print("Saved to epoch2_evaluation.csv")

Evaluated examples: 50
Saved to epoch2_evaluation.csv


In [62]:
print("Total examples:", len(evaluation_epoch2))

print("\nAverage response length:")
print(
    "Base model:",
    evaluation_epoch2["base_response"].str.len().mean()
)
print(
    "Epoch-2 model:",
    evaluation_epoch2["epoch2_response"].str.len().mean()
)

print("\nSample comparisons:")

for i in range(5):
    row = evaluation_epoch2.iloc[i]

    print(f"\n--- Example {i+1} ---")
    print("Question:", row["instruction"])
    print("Expected:", row["expected"])
    print("Base:", row["base_response"])
    print("Epoch 2:", row["epoch2_response"])

Total examples: 50

Average response length:
Base model: 374.42
Epoch-2 model: 235.2

Sample comparisons:

--- Example 1 ---
Question: Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?
Expected: Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree
Base: In the fantasy series "A Song of Ice and Fire" by George R.R. Martin, there is no character named "Garth Greenhand" or "The High King of the First Men." The characters you might be thinking of could include:
- Aria (a young girl)
- Joffrey Baratheon
- Sansa Linenweaver
- Arya Stark
- Sansa's mother, who was probably also named Sansa
- Sansa's father, who
Epoch 2: The children of the legendary Garth Greenhand, the High King of the First Men, were named Gw

In [63]:
import pandas as pd

epoch1 = pd.read_csv("test_evaluation.csv")

comparison = pd.DataFrame({
    "question": epoch1["instruction"],
    "expected": epoch1["expected"],
    "epoch1": epoch1["finetuned_response"],
    "epoch2": evaluation_epoch2["epoch2_response"]
})

print(comparison[[
    "question", "epoch1", "epoch2"
]].head(10).to_string(index=False))

                                                                                                                    question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              epoch1                                                                                                                                                                                                                                                                                                                                                                                        

In [64]:
!pip install rouge-score

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24997 sha256=6f55bc3a4075aab3bcfd400b59c409e073458bf29c7a4439660413044587a1ee
  Stored in directory: /home/mohsinkhan/.cache/pip/wheels/fa/77/62/e75e60437a9821bbc6523332a7453f56b0720535477de63f75
Successfully built rouge-score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rouge-score]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [65]:
from rouge_score import rouge_scorer
import pandas as pd

scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

def get_rouge(expected, prediction):
    return scorer.score(
        str(expected),
        str(prediction)
    )["rougeL"].fmeasure

epoch1 = pd.read_csv("test_evaluation.csv")

epoch1_score = epoch1.apply(
    lambda row: get_rouge(
        row["expected"],
        row["finetuned_response"]
    ),
    axis=1
).mean()

epoch2_score = evaluation_epoch2.apply(
    lambda row: get_rouge(
        row["expected"],
        row["epoch2_response"]
    ),
    axis=1
).mean()

print(f"Epoch 1 ROUGE-L: {epoch1_score:.4f}")
print(f"Epoch 2 ROUGE-L: {epoch2_score:.4f}")

Epoch 1 ROUGE-L: 0.2501
Epoch 2 ROUGE-L: 0.2765


In [66]:
print("Final training loss:", trainer.state.log_history[-1])

Final training loss: {'train_runtime': 3601.2788, 'train_samples_per_second': 6.414, 'train_steps_per_second': 0.802, 'total_flos': 8421688413186048.0, 'train_loss': 2.1016977381508104, 'entropy': 1.9979452010123961, 'mean_token_accuracy': 0.5616156824173466, 'num_tokens': 5873868.0, 'epoch': 2.0, 'step': 2888}


In [67]:
from collections import Counter

print(Counter(train_dataset["category"]))

Counter({'open_qa': 3031, 'general_qa': 1738, 'classification': 1683, 'closed_qa': 1424, 'brainstorming': 1419, 'information_extraction': 1203, 'summarization': 946, 'creative_writing': 564})


In [68]:
import pandas as pd
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rougeL"], use_stemmer=True
)

def rouge_l(expected, prediction):
    return scorer.score(
        str(expected), str(prediction)
    )["rougeL"].fmeasure

# Add scores to the existing comparison
comparison["epoch1_rougeL"] = comparison.apply(
    lambda r: rouge_l(r["expected"], r["epoch1"]), axis=1
)

comparison["epoch2_rougeL"] = comparison.apply(
    lambda r: rouge_l(r["expected"], r["epoch2"]), axis=1
)

comparison["improvement"] = (
    comparison["epoch2_rougeL"] - comparison["epoch1_rougeL"]
)

print("Epoch 2 improved:", (comparison["improvement"] > 0).sum())
print("Epoch 2 unchanged:", (comparison["improvement"] == 0).sum())
print("Epoch 2 declined:", (comparison["improvement"] < 0).sum())

Epoch 2 improved: 27
Epoch 2 unchanged: 14
Epoch 2 declined: 9


In [71]:
for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(entry)

In [72]:
split = train_dataset.train_test_split(
    test_size=0.05,
    seed=42
)

train_data = split["train"]
validation_data = split["test"]

print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))
print("Final test examples:", len(test_dataset))

Training examples: 11407
Validation examples: 601
Final test examples: 3003


In [73]:
print("Trainer class:", type(trainer).__name__)
print("Epochs:", trainer.args.num_train_epochs)
print("Evaluation strategy:", trainer.args.eval_strategy)
print("Save strategy:", trainer.args.save_strategy)
print("Current global step:", trainer.state.global_step)

Trainer class: SFTTrainer
Epochs: 2
Evaluation strategy: IntervalStrategy.NO
Save strategy: SaveStrategy.EPOCH
Current global step: 2888


In [74]:
print(trainer.args)

SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=True,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=F

In [75]:
trainer.model.save_pretrained("./dolly-lora/epoch2_backup")
tokenizer.save_pretrained("./dolly-lora/epoch2_backup")

('./dolly-lora/epoch2_backup/tokenizer_config.json',
 './dolly-lora/epoch2_backup/chat_template.jinja',
 './dolly-lora/epoch2_backup/tokenizer.json')

In [76]:
print("Model:", type(trainer.model).__name__)
print("Training dataset size:", len(trainer.train_dataset))
print("Dataset columns:", trainer.train_dataset.column_names)
print("Original training config:", type(trainer.args).__name__)

Model: PeftModelForCausalLM
Training dataset size: 11550
Dataset columns: ['prompt', 'completion', 'input_ids', 'labels']
Original training config: SFTConfig


In [77]:
print(trainer.train_dataset[0].keys())

print("\nPrompt:")
print(trainer.train_dataset[0]["prompt"][:500])

print("\nCompletion:")
print(trainer.train_dataset[0]["completion"][:500])

dict_keys(['prompt', 'completion', 'input_ids', 'labels'])

Prompt:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a digital camera?<|im_end|>
<|im_start|>assistant


Completion:
A digital camera is a hand held device that captures images with a push of a button.  These images are usually stored on a memory disk or card which can be downloaded to a computer at a later time.  Digital cameras were also part of a revolution that replaced photographic film.  Digital cameras use flashes when no natural light is available.  They produce better picture quality than traditional photographic film cameras and there is no cost to develop pictures and print them in order to enjoy th


In [78]:
print("Original train dataset:", len(train_dataset))
print("New training split:", len(train_data))
print("Validation split:", len(validation_data))

print("\nOriginal columns:", train_dataset.column_names)
print("New training columns:", train_data.column_names)
print("Validation columns:", validation_data.column_names)

Original train dataset: 12008
New training split: 11407
Validation split: 601

Original columns: ['instruction', 'context', 'response', 'category']
New training columns: ['instruction', 'context', 'response', 'category']
Validation columns: ['instruction', 'context', 'response', 'category']
